# ตอนที่ 4: DPO — เมื่อโมเดลภาษากลายเป็น reward model ของตัวเอง

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/04_dpo.ipynb)

โน้ตบุ๊กประกอบบทความ **[LLM 4/10] DPO: เมื่อโมเดลภาษากลายเป็น reward model ของตัวเอง**

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com/blog/llm-04-dpo)

---

## โครงของโน้ตบุ๊กนี้ (เรียงตรงกับหัวข้อในบทความ)

1. ปัญหา (Problem statement)
2. เราจะทำอะไร (Solution)
3. สมการ (Equation)
4. เห็นภาพสมการ (Visualize)
5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline ก่อนแตะอะไรทั้งสิ้น
6. เตรียมข้อมูล (Data)
7. โค้ดหลัก (Main code) — **เขียน DPO loss เองด้วยมือ แล้วพิสูจน์ว่าตรงกับ TRL**
8. ผลลัพธ์ (Results)
9. เปรียบเทียบ (Comparison)
10. สรุป (Summary)

---

## หัวใจของโน้ตบุ๊กนี้

หัวข้อที่ 7 มีบรรทัดนี้อยู่:

```python
assert torch.allclose(loss_manual, loss_trl, atol=1e-4)
```

`loss_manual` มาจาก DPO loss ที่เราเขียนเองราว 25 บรรทัดด้วย PyTorch ล้วน ๆ
ส่วน `loss_trl` มาจาก `DPOTrainer` ของไลบรารี TRL ที่คนทั้งโลกใช้ บน batch เดียวกันเป๊ะ

บทความ tutorial ส่วนใหญ่จบที่ "เรียกไลบรารีแล้วมันก็ทำงาน" แล้วขอให้คุณเชื่อ
บรรทัดนั้นทำให้โน้ตบุ๊กนี้ **พิสูจน์ตัวเอง** แทนที่จะขอความเชื่อ
ถ้ามันผ่าน แปลว่าสมการที่อนุมานในหัวข้อ 3 ให้ค่าเท่ากับของจริง
และคุณ implement DPO เองได้แล้ว ไม่ใช่แค่เรียกใช้เป็น

---

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
  เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรันเซลล์แรก
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงกำหนด `torch_dtype=torch.float16`, `attn_implementation="sdpa"` และ `fp16=True` เอง
  อย่างชัดเจนทุกครั้ง (รายละเอียดอยู่ในคอมเมนต์ของ Cell 0 — อ่านให้ครบ)
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  (120 ข้อ, 4 slice) เพื่อให้ตัวเลขของแต่ละตอนเทียบกันได้จริง
- **ห้ามเทรนบน KobEval-TH เด็ดขาด** ชุดนี้เป็นชุดวัดผลอย่างเดียว
  ข้อมูลที่ใช้เทรนในตอนนี้มาจาก `iapp/dpo_thai_tutorial` และ
  `airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k` ซึ่งแยกขาดจากชุดวัดผล

## เวลาที่ใช้โดยประมาณบน T4 ฟรี (รวมเวลาโหลดโมเดล)

| ขั้นตอน | เวลาโดยประมาณ |
|---|---|
| Cell 0 ติดตั้ง dependencies | ~2 นาที |
| Cell 1 โหลดโมเดล + วัด baseline (TH-INSTR 30 ข้อ) | ~3 นาที |
| สร้างคู่ preference (generate rejected) | ~3 นาที |
| พิสูจน์ว่า loss ที่เขียนเอง == TRL | ~1 นาที |
| เทรน DPO 2 epoch | ~8 นาที |
| วัดผลซ้ำ + วาดกราฟ | ~5 นาที |
| **รวม** | **~20-22 นาที** |

ถ้าอยากให้จบเร็วขึ้น ลด `N_AUG` (จำนวนคู่ที่สร้างเพิ่ม) ในหัวข้อที่ 6


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
REPO_DIR = "/content/thai-llm-tutorials"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

# ลดการแตกกระจายของหน่วยความจำ (fragmentation) -- ต้องตั้งก่อน torch แตะ CUDA
# T4 มี VRAM จำกัด พอ tensor ก้อนใหญ่ถูกจอง/คืนสลับกัน จะเกิดช่องว่างที่ใช้ต่อไม่ได้
# จน OOM ทั้งที่ยังเหลือที่รวม ๆ อยู่ (ข้อความ error ของ PyTorch เองก็แนะนำค่านี้)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# bf16 "แบบมีฮาร์ดแวร์รองรับจริง" เริ่มที่ Ampere (sm_80) ขึ้นไปเท่านั้น
#
# หมายเหตุสำคัญ: torch.cuda.is_bf16_supported() ตอบ True บน T4 (sm_75) ด้วย
# ตั้งแต่ torch รุ่นใหม่ ๆ เพราะมันนับ "การจำลอง (emulation)" ว่ารองรับด้วย
# ซึ่งช้ากว่า fp16 มากและไม่ใช่สิ่งที่เราต้องการ เราจึงเช็ค compute capability
# ตรง ๆ แทน -- นี่คือกับดักจริงที่เจอตอนรันบน Colab
NATIVE_BF16 = CAPABILITY[0] >= 8
SUPPORTS_BF16 = NATIVE_BF16          # ใช้ชื่อเดิมต่อได้ แต่ความหมายคือ "native"

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}   (native; torch บอกว่า "
      f"{torch.cuda.is_bf16_supported()} เพราะนับ emulation ด้วย)")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False   (native)
# ทั้งที่ torch.cuda.is_bf16_supported() ตอบ True -- อย่าเชื่อค่านั้น
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
#    pip install -e ลงทะเบียนแพ็กเกจไว้แล้ว แต่ sys.path ของเคอร์เนลที่รันอยู่
#    อาจยังไม่เห็น จึงใส่ path ของ repo เข้าไปตรง ๆ กันพลาด (idempotent)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums
from kobeval.plotting import use_thai_font

# 8) ตั้งฟอนต์ไทยให้ matplotlib "ครั้งเดียวตรงนี้" มีผลกับทุกกราฟในโน้ตบุ๊ก
#    ทำไมต้องมีขั้นตอนนี้: apt ติดตั้งฟอนต์ใน (2) หลังจาก matplotlib สร้าง
#    แคชรายชื่อฟอนต์ไปแล้ว มันจึงมองไม่เห็นฟอนต์ไทย และวาดออกมาเป็นกล่อง []
#    use_thai_font() จะสแกนดิสก์ใหม่แล้วลงทะเบียนฟอนต์ให้
_thai_font = use_thai_font()
print(f"Thai font      : {_thai_font or 'ไม่พบ! กราฟภาษาไทยจะเป็นกล่อง [] -- รัน apt-get ในข้อ (2) ใหม่'}")

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")


---

## 1. ปัญหา (Problem statement)

สมมติคุณอยากให้ผู้ช่วย AI ของคุณ "ตอบเป็นภาษาไทยเสมอ" — ฟังดูง่าย
แต่ลองเขียนเป็น loss function ดูสิครับ คุณจะเขียนไม่ออก

นี่คือปัญหาแกนกลางของ alignment: คุณภาพของคำตอบ **เขียนเป็นสมการไม่ได้**
"สุภาพกว่า" "เป็นธรรมชาติกว่า" "ไม่หลุดไปเป็นภาษาอังกฤษ" ไม่มีเฉลยเดียวที่ถูกต้อง
มีแต่การ **เปรียบเทียบ** ข้อมูลจึงมีหน้าตาเป็นสามสิ่ง:
prompt $x$, คำตอบที่ชอบ $y_w$ (chosen), คำตอบที่ไม่ชอบ $y_l$ (rejected)

RLHF แบบ PPO แก้ปัญหานี้ด้วยการเดินอ้อมสองขั้น คือเทรน reward model แล้วใช้ RL
ซึ่งมีราคาที่ต้องจ่าย:

| ปัญหาของ RLHF/PPO | ผลที่เกิดขึ้นจริง |
|---|---|
| ต้องเทรนโมเดลเพิ่ม 1 ตัว | เพิ่มขั้นตอน เพิ่มโอกาสพัง เพิ่มเวลา |
| ต้องโหลด 4 โมเดลพร้อมกัน | policy + ref + reward + value — VRAM บาน |
| reward hacking | โมเดลหาช่องโกงคะแนน โดยที่มนุษย์ไม่ได้ชอบขึ้นเลย |
| PPO ไวต่อ hyperparameter | รันสองรอบด้วย seed ต่างกัน อาจได้ผลคนละเรื่อง |

คำถามของตอนนี้: **เราข้ามสองขั้นนั้นไปเลยได้ไหม**

---

## 2. เราจะทำอะไร (Solution)

ได้ และเหตุผลสวยมาก

objective ของ RLHF ที่มี KL constraint **มีคำตอบในรูปปิด (closed form)**
เรารู้อยู่แล้วว่า policy ที่ดีที่สุดหน้าตาเป็นอย่างไร โดยไม่ต้องรัน RL แม้แต่ step เดียว
เมื่อรู้แบบนั้น เราก็ **พลิกสมการกลับด้าน** — แทนที่จะถามว่า "reward นี้ให้ policy อะไร"
เราถามว่า "policy นี้แปลว่า reward เท่าไหร่"

**เมื่อพลิกสมการ ตัวโมเดลภาษาเองก็คือ reward model อยู่แล้วโดยปริยาย**
reward model กับ RL loop ไม่ได้ถูก "ประมาณทิ้ง" แต่มัน **ตัดกันหายไปทางพีชคณิต**

นี่คือ **DPO (Direct Preference Optimization)** โดย Rafailov และคณะ (2023)


---

## 3. สมการ (Equation)

### 3.1 ตั้งโจทย์: RLHF objective

$$
\max_{\pi}\ \mathbb{E}_{x\sim\mathcal{D},\,y\sim\pi(\cdot|x)}\big[r(x,y)\big]
- \beta\,\mathbb{D}_{\text{KL}}\big[\pi(y|x)\,\|\,\pi_{\text{ref}}(y|x)\big]
$$

อ่านเป็นภาษาคน: **"ทำ reward ให้สูงที่สุด แต่ห้ามเดินห่างจากโมเดลตั้งต้นมากเกินไป"**

### 3.2 ขั้นที่ 1 — คำตอบในรูปปิด

$$
\pi^*(y|x) = \frac{1}{Z(x)}\pi_{\text{ref}}(y|x)\exp\left(\frac{1}{\beta}r(x,y)\right)
$$

$Z(x)$ ขึ้นกับ $x$ เท่านั้น **ไม่ขึ้นกับ $y$** — จำประโยคนี้ไว้ เดี๋ยวมันจะเป็นพระเอก
ในทางปฏิบัติเราคำนวณ $Z(x)$ ไม่ได้ เพราะต้องรวมทุกคำตอบที่เป็นไปได้ทั้งจักรวาล

### 3.3 ขั้นที่ 2 — พลิกสมการหา reward

$$
r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta\log Z(x)
$$

**reward ทุกฟังก์ชันเขียนใหม่ได้ในรูปของ policy สองตัว** ไม่ต้องเทรน reward model ใด ๆ

### 3.4 ขั้นที่ 3 — แทนใน Bradley-Terry แล้ว $Z(x)$ ตัดหาย

$$
p(y_w \succ y_l \mid x) = \sigma\big(r(x,y_w) - r(x,y_l)\big)
$$

reward ปรากฏในรูป **ผลต่าง** เท่านั้น พจน์ $\beta\log Z(x)$ จึงอยู่ทั้งสองฝั่งเท่ากันเป๊ะ
เพราะเป็น $x$ ตัวเดียวกัน มันจึง **ตัดกันหายไป** เหลือ

$$
\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y_w,y_l)}\left[\log\sigma\left(
\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}
- \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]
$$

สิ่งที่คำนวณไม่ได้หายไปแล้ว ที่เหลือคือ log-probability ของโมเดลสองตัว
บนข้อความที่เรามีอยู่แล้ว ซึ่งได้จาก forward pass ธรรมดา
**ไม่มีการสุ่มคำตอบ ไม่มี rollout ไม่มี value function** — DPO เป็น supervised learning เต็มตัว

### 3.5 Gradient — ที่ซึ่งสัญชาตญาณอยู่

$$
\nabla_\theta\mathcal{L}_{\text{DPO}} = -\beta\,\mathbb{E}\left[\sigma(\hat r_l - \hat r_w)
\left(\nabla_\theta\log\pi_\theta(y_w|x) - \nabla_\theta\log\pi_\theta(y_l|x)\right)\right]
$$

โดย $\hat r = \beta\log(\pi_\theta/\pi_{\text{ref}})$ เรียกว่า **implicit reward**

- วงเล็บขวา = **ทิศทาง** ดัน log-prob ของ $y_w$ ขึ้น และกด $y_l$ ลง พร้อมกัน
- $\sigma(\hat r_l - \hat r_w)$ = **น้ำหนัก** คือ "โมเดลจัดอันดับคู่นี้ผิดแค่ไหน"

ถ้าโมเดลจัดอันดับคู่นี้ถูกอยู่แล้ว น้ำหนักจะเข้าใกล้ศูนย์ และคู่นั้นแทบไม่ส่ง gradient เลย
**DPO จึงโฟกัสไปที่ความผิดพลาดของตัวเองโดยอัตโนมัติ**

---

## 4. เห็นภาพสมการ (Visualize)

รูปของหัวข้อนี้ถูกวาดในเซลล์ **ถัดจาก Cell 1** เพราะกติกาของซีรีส์กำหนดว่า
เซลล์โค้ดที่สองของทุกตอนต้องเป็นการวัด baseline เสมอ ห้ามมีอะไรมาแทรกก่อน

กราฟจะแสดง loss กับน้ำหนัก gradient ที่ค่า $\beta$ ต่างกัน
สังเกตแผงขวา: เมื่อ margin เป็นบวกมาก ๆ น้ำหนักลู่เข้าศูนย์ คู่นั้น "เรียนจบแล้ว"
ยิ่ง $\beta$ สูง เส้นยิ่งชัน คือทั้งเรียนเร็วและเลิกเรียนเร็ว


---

## 5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ**

เซลล์ถัดไปคือ Cell 1 ซึ่งวัดผลโมเดลตั้งต้นด้วย KobEval-TH ก่อนที่เราจะเทรนอะไรทั้งนั้น
สังเกตค่า `th_ratio` เป็นพิเศษ — มันคือสัดส่วนตัวอักษรไทยในคำตอบ
และเป็น metric ที่ DPO ในตอนนี้ตั้งใจจะขยับโดยตรง เพราะ `rejected` ทุกตัว
คือคำตอบที่โมเดลฐานสร้างเอง ซึ่งมักไหลไปเป็นภาษาอังกฤษกลางประโยค

> **หมายเหตุเรื่องเวลา:** เรารัน KobEval-TH เฉพาะ slice `TH-INSTR` (30 ข้อ)
> เพราะ DPO ในตอนนี้แก้เรื่อง "ตอบตามที่สั่งและตอบเป็นภาษาไทย" ซึ่ง TH-INSTR วัดตรงที่สุด
> เพิ่ม slice อื่นใน `KOBEVAL_SLICES` ได้ถ้ามีเวลา — สัญญาการวัดผลเหมือนกันทุกประการ


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนเทรน/ก่อนแก้อะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# =============================================================================
import gc
import json
import math
import time

import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
KOBEVAL_SLICES = ["TH-INSTR"]
DEV = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ค่า auto เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- sdpa เท่านั้น
    device_map=DEV,
)
model.eval()

cfg = model.config
print(f"โหลดโมเดลแล้ว: {MODEL_ID}")
print(f"dtype จริงของ parameter: {next(model.parameters()).dtype}")
print(f"จำนวนพารามิเตอร์: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print("--- ค่าจริงจาก config ของ Qwen3-0.6B (ไม่ใช่ค่าที่จำมา) ---")
print(f"  hidden_size             : {cfg.hidden_size}")
print(f"  intermediate_size       : {cfg.intermediate_size}")
print(f"  num_hidden_layers       : {cfg.num_hidden_layers}")
print(f"  num_attention_heads     : {cfg.num_attention_heads}")
print(f"  num_key_value_heads     : {cfg.num_key_value_heads}")
print(f"  head_dim                : {getattr(cfg, 'head_dim', None)}")
print(f"  vocab_size              : {cfg.vocab_size}")
print(f"  max_position_embeddings : {cfg.max_position_embeddings}")

VRAM_AFTER_BASE = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"\nVRAM ที่ใช้หลังโหลดโมเดลฐาน: {VRAM_AFTER_BASE:.3f} GB")

# รันเบนช์มาร์กกลาง -- สัญญาการวัดผลถูกตรึงไว้ใน kobeval แล้ว
# (greedy, max_new_tokens=256, enable_thinking=False, seed=42)
t0 = time.time()
baseline = evaluate(
    model,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name="Qwen3-0.6B (baseline)",
    out_path="results_baseline.json",
)
print(f"\nใช้เวลาวัด baseline: {time.time() - t0:.0f} วินาที")

for name, s in baseline["slices"].items():
    print(
        f"{name:9s} acc={s['accuracy']:.3f} "
        f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
        f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}"
    )

print(f"\nรวมทุก slice: acc={baseline['overall']['accuracy']:.3f}  "
      f"th_ratio={baseline['overall']['th_ratio']:.2f}")


---

### รูปประกอบหัวข้อที่ 4 — DPO loss และน้ำหนัก gradient

เซลล์ถัดไปไม่ใช้ GPU และไม่ใช้เน็ต วาดจากสมการ 3.4 และ 3.5 ตรง ๆ
ค่า $\ln 2 \approx 0.693$ ที่เส้นประชี้ไว้ คือ loss ตอนที่โมเดล "ไม่มีความเห็น"
และเป็นค่าที่เราจะเห็นจริงในหัวข้อ 7 ตอนที่ policy กับ reference ยังเหมือนกันเป๊ะ


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 4 -- DPO loss และน้ำหนัก gradient เทียบกับ margin ที่ค่า beta ต่างกัน
# เซลล์นี้ไม่ใช้ GPU และไม่ใช้เน็ต
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

from kobeval import use_thai_font

use_thai_font()


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


delta = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))

for beta, color in zip([0.1, 0.3, 1.0], ["#2563eb", "#f59e0b", "#dc2626"]):
    loss = -np.log(sigmoid(beta * delta))
    weight = sigmoid(-beta * delta)
    axes[0].plot(delta, loss, lw=2, color=color, label=f"β = {beta}")
    axes[1].plot(delta, weight, lw=2, color=color, label=f"β = {beta}")

axes[0].axhline(np.log(2), ls=":", color="#64748b", lw=1)
axes[0].text(-5.8, np.log(2) + 0.15, "ln 2 = จุดที่โมเดลไม่มีความเห็น", fontsize=8, color="#475569")
axes[0].set_xlabel("margin Δ = (log π/π_ref ของ chosen) − (ของ rejected)")
axes[0].set_ylabel("DPO loss")
axes[0].set_title("loss ลดลงเมื่อ margin กว้างขึ้น", fontsize=11)
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_axisbelow(True)

axes[1].set_xlabel("margin Δ")
axes[1].set_ylabel("น้ำหนักของคู่นี้ใน gradient")
axes[1].set_title("คู่ที่จัดอันดับถูกแล้ว แทบไม่มี gradient เหลือ", fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_axisbelow(True)

fig.tight_layout()
fig.savefig("fig_dpo_loss_and_gradient.png", dpi=150, bbox_inches="tight")
plt.show()

print("บันทึกรูปแล้ว: fig_dpo_loss_and_gradient.png")
print("ที่ Δ = 0 (โมเดลไม่มีความเห็น) loss = ln 2 =", round(float(np.log(2)), 6))
print("นี่คือค่าที่เราจะเห็นจริงในหัวข้อ 7 ตอนที่ policy กับ reference ยังเหมือนกันเป๊ะ")


### reference model ที่ไม่กิน VRAM เพิ่มเลยแม้แต่ไบต์เดียว

DPO ต้องใช้ทั้ง $\pi_\theta$ และ $\pi_{\text{ref}}$ ฟังดูเหมือนต้องโหลดโมเดลสองตัว
ซึ่งบน T4 ฟรีคือจุดที่คนส่วนใหญ่เจอ OOM แล้วเลิก

ทางออกคือเทรนด้วย **LoRA**: `policy` คือโมเดลฐาน + adapter ส่วน $\pi_{\text{ref}}$
คือ **โมเดลฐานตัวเดิมที่ปิด adapter ไว้** เรียกผ่าน context manager
`policy.disable_adapter()` ได้เลย

น้ำหนักฐานถูกใช้ร่วมกันทั้งสองบทบาท ต้นทุน VRAM ของ reference model จึงเป็น
**ศูนย์ไบต์** เซลล์ถัดไปพิมพ์ตัวเลข VRAM ก่อนและหลังติด adapter ออกมาให้เห็นกับตา
นี่ไม่ใช่แค่ลูกเล่นประหยัดหน่วยความจำ มันคือเหตุผลเชิงสถาปัตยกรรมที่ทำให้
ตอนนี้รันได้บนการ์ดฟรี

> **หมายเหตุ:** ในลำดับของซีรีส์ ตอนที่ 2 จะส่ง LoRA adapter ที่ผ่าน SFT มาแล้วให้ตอนนี้
> ต่อยอด แต่โน้ตบุ๊กนี้ตั้งใจให้ **รันจบได้ด้วยตัวเอง** จึงสร้าง adapter ใหม่จาก
> `Qwen/Qwen3-0.6B` ตรง ๆ ไม่อ้างอิงไฟล์ภายนอกที่ยังไม่มีอยู่จริง

### กับดัก fp16 ที่ต้องปิดก่อนเริ่ม

`fp16=True` แปลว่า *mixed precision* ไม่ได้แปลว่าน้ำหนักที่ optimizer อัปเดตเป็น fp16
ถ้าพารามิเตอร์ที่เทรนได้ (คือ LoRA adapter) เป็น fp16 `GradScaler` จะโยน

```
ValueError: Attempting to unscale FP16 gradients.
```

วิธีแก้คือ cast **เฉพาะพารามิเตอร์ที่เทรนได้** เป็น fp32 ส่วนน้ำหนักฐาน 596M ตัว
ยังเป็น fp16 เหมือนเดิม — VRAM แทบไม่ขยับ เพราะ adapter มีขนาดไม่ถึง 1% ของโมเดล


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 5.1 -- ติด LoRA adapter: policy = base + adapter, reference = base (ปิด adapter)
# -----------------------------------------------------------------------------
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,      # ตั้งเป็น 0 เพื่อให้ผลลัพธ์เป็น deterministic ตอนพิสูจน์ในหัวข้อ 7
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

policy = get_peft_model(model, lora_cfg)

# cast เฉพาะพารามิเตอร์ที่เทรนได้เป็น fp32 (ดูเหตุผลด้านบน)
def cast_trainable_to_fp32(m):
    n_cast = 0
    for _, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
            n_cast += 1
    return n_cast


n_cast = cast_trainable_to_fp32(policy)

VRAM_AFTER_LORA = torch.cuda.memory_allocated() / (1024 ** 3)
n_train = sum(p.numel() for p in policy.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in policy.parameters())

print(f"พารามิเตอร์ที่เทรน : {n_train / 1e6:.2f}M จาก {n_total / 1e6:.1f}M "
      f"({100 * n_train / n_total:.2f}%)")
print(f"cast เป็น fp32 ไป  : {n_cast} tensors")
print()
print(f"VRAM ก่อนติด adapter : {VRAM_AFTER_BASE:.3f} GB")
print(f"VRAM หลังติด adapter : {VRAM_AFTER_LORA:.3f} GB")
print(f"ส่วนต่าง             : {VRAM_AFTER_LORA - VRAM_AFTER_BASE:.3f} GB")
print()
print("และนี่คือประเด็นสำคัญ: reference model ไม่ได้กิน VRAM เพิ่มอีกเลยแม้แต่ไบต์เดียว")
print("เพราะมันคือ 'โมเดลฐานตัวเดิม' ที่เราปิด adapter ชั่วคราวเท่านั้น")

policy.eval()
with torch.no_grad():
    probe = tokenizer("สวัสดีครับ วันนี้อากาศ", return_tensors="pt").to(DEV)
    logits_on = policy(**probe).logits
    with policy.disable_adapter():
        logits_off = policy(**probe).logits

same = torch.allclose(logits_on, logits_off)
print()
print(f"ตอนนี้ policy กับ reference ให้ logits เท่ากันหรือไม่: {same}")
print("ควรได้ True เพราะ LoRA ถูก init ให้ B = 0 ทำให้ adapter ยังไม่เปลี่ยนอะไรเลย")
print("จำข้อนี้ไว้ให้ดี เดี๋ยวมันจะสำคัญมากตอนพิสูจน์ในหัวข้อ 7.2")


---

## 6. เตรียมข้อมูล (Data)

ข้อมูล DPO ต้องมี 3 คอลัมน์เท่านั้น: `prompt`, `chosen`, `rejected`

**ชุดที่ 1 — `iapp/dpo_thai_tutorial`** (100 คู่, Apache-2.0)
เป็นชุดข้อมูลที่ **ผมทำขึ้นเองสำหรับซีรีส์นี้** และปล่อยเป็น Apache-2.0 ให้เอาไปใช้ต่อได้
เป็นคู่ preference ภาษาไทยที่คัดด้วยมือ เน้นความสุภาพและความเป็นธรรมชาติของภาษา

**ชุดที่ 2 — สร้างเองราว 400 คู่** จาก
`airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k` ด้วยกติกาที่ตรงไปตรงมามาก:

- `chosen` = คำตอบอ้างอิงภาษาไทยที่มากับชุดข้อมูล
- `rejected` = **คำตอบที่โมเดลฐานสร้างเองด้วย greedy decoding**

> **ทำไม rejected ที่โมเดลสร้างเองถึงดีกว่า rejected ที่ไปหามา**
> คำตอบ greedy ของ Qwen3-0.6B บน prompt ภาษาไทยมักไหลไปเป็นภาษาอังกฤษกลางประโยค
> นั่นคือข้อบกพร่อง **จริง** ของโมเดลตัวนี้ ไม่ใช่ข้อบกพร่องที่เราสมมติขึ้น
> การใช้มันเป็น `rejected` ทำให้ gradient ชี้ตรงไปที่พฤติกรรมที่เราอยากแก้พอดี
> ถ้าไปเอา rejected จากโมเดลอื่นมา คุณจะกำลังสอนให้โมเดล "ไม่เป็นโมเดลอื่น"
> ซึ่งไม่ใช่สิ่งที่คุณต้องการ

รวมได้ราว **500 คู่** แบ่ง held-out ไว้ 15% สำหรับวัดผล **ไม่แตะระหว่างเทรน**

หมายเหตุทางเทคนิคของเซลล์ถัดไป: ตอน generate เราตั้ง `padding_side = "left"`
เพราะโมเดล decoder-only ต่อข้อความจากตำแหน่งสุดท้ายของ input
ถ้า pad ทางขวา ตำแหน่งสุดท้ายจะเป็น `<pad>` แล้วคำตอบจะเพี้ยนทั้ง batch
พอ generate เสร็จเราตั้งกลับเป็น `"right"` เพราะ `DPOTrainer` ต้องการแบบนั้น


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6 -- ประกอบชุดข้อมูล preference ภาษาไทย
# -----------------------------------------------------------------------------
from datasets import Dataset, load_dataset

N_AUG = 400            # จำนวนคู่ที่สร้างเพิ่มเอง (ลดลงได้ถ้าอยากให้จบเร็ว)
GEN_BATCH = 16
GEN_MAX_NEW = 96
HELDOUT_FRAC = 0.15


def pick_column(columns, candidates):
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


def to_prompt(user_text):
    """แปลงคำถามดิบเป็น prompt ตาม chat template ของ Qwen3 (ปิด thinking mode)"""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


# --- ชุดที่ 1: ของ Kobkrit เอง Apache-2.0 ---
seed_ds = load_dataset("iapp/dpo_thai_tutorial", split="train")
print(f"iapp/dpo_thai_tutorial columns: {seed_ds.column_names}  n={len(seed_ds)}")

p_col = pick_column(seed_ds.column_names, ["prompt", "question", "instruction", "input"])
c_col = pick_column(seed_ds.column_names, ["chosen", "preferred", "good", "response_j"])
r_col = pick_column(seed_ds.column_names, ["rejected", "dispreferred", "bad", "response_k"])
assert None not in (p_col, c_col, r_col), f"schema ไม่ตรงที่คาด: {seed_ds.column_names}"

pairs = []
for row in seed_ds:
    prompt, chosen, rejected = row[p_col], row[c_col], row[r_col]
    if not all(isinstance(v, str) and v.strip() for v in (prompt, chosen, rejected)):
        continue
    pairs.append({
        "prompt": to_prompt(prompt.strip()) if not prompt.startswith("<|im_start|>") else prompt,
        "chosen": chosen.strip(),
        "rejected": rejected.strip(),
        "source": "iapp/dpo_thai_tutorial",
    })
print(f"ได้คู่จากชุดที่ 1: {len(pairs)}")

# --- ชุดที่ 2: สร้าง rejected ด้วยตัวโมเดลฐานเอง ---
aug_ds = load_dataset("airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k", split="train")
print(f"wangchanx columns: {aug_ds.column_names}  n={len(aug_ds)}")

a_q = pick_column(aug_ds.column_names, ["instruction", "question", "input", "prompt"])
a_a = pick_column(aug_ds.column_names, ["output", "response", "answer", "completion"])
assert None not in (a_q, a_a), f"schema ไม่ตรงที่คาด: {aug_ds.column_names}"

aug_ds = aug_ds.shuffle(seed=SEED).select(range(min(N_AUG * 2, len(aug_ds))))
cand = []
for row in aug_ds:
    q, a = row[a_q], row[a_a]
    if not (isinstance(q, str) and isinstance(a, str)):
        continue
    q, a = q.strip(), a.strip()
    if 10 <= len(q) <= 400 and 30 <= len(a) <= 900:
        cand.append((q, a))
    if len(cand) >= N_AUG:
        break
print(f"คำถามที่ผ่านการคัดกรอง: {len(cand)}")

policy.eval()
tokenizer.padding_side = "left"      # decoder-only ต้อง pad ซ้ายตอน generate
t0 = time.time()
generated = []
with torch.no_grad(), policy.disable_adapter():   # ให้เป็นคำตอบของ "โมเดลฐาน" จริง ๆ
    for i in range(0, len(cand), GEN_BATCH):
        chunk = cand[i:i + GEN_BATCH]
        prompts = [to_prompt(q) for q, _ in chunk]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=320).to(DEV)
        torch.manual_seed(SEED)
        out = policy.generate(
            **enc,
            max_new_tokens=GEN_MAX_NEW,
            do_sample=False,                       # greedy -- ทำซ้ำได้
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = out[:, enc["input_ids"].shape[1]:]
        generated.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
        if (i // GEN_BATCH) % 5 == 0:
            print(f"  generate {i + len(chunk)}/{len(cand)} "
                  f"({time.time() - t0:.0f}s)")
tokenizer.padding_side = "right"     # DPOTrainer ต้องการ padding ทางขวา

print(f"generate เสร็จใน {time.time() - t0:.0f} วินาที")

n_added = 0
for (q, a), gen_text in zip(cand, generated):
    gen_text = gen_text.strip()
    if len(gen_text) < 10 or gen_text == a:
        continue
    pairs.append({
        "prompt": to_prompt(q),
        "chosen": a,
        "rejected": gen_text,
        "source": "wangchanx + base-model greedy",
    })
    n_added += 1
print(f"ได้คู่จากชุดที่ 2: {n_added}")

random.Random(SEED).shuffle(pairs)
n_heldout = int(round(HELDOUT_FRAC * len(pairs)))
heldout_pairs, train_pairs = pairs[:n_heldout], pairs[n_heldout:]

train_ds = Dataset.from_list(
    [{k: p[k] for k in ("prompt", "chosen", "rejected")} for p in train_pairs]
)

len_chosen = [len(tokenizer(p["chosen"], add_special_tokens=False).input_ids) for p in pairs]
len_rejected = [len(tokenizer(p["rejected"], add_special_tokens=False).input_ids) for p in pairs]
DATA_LEN_STATS = {
    "mean_chosen_tokens": sum(len_chosen) / len(len_chosen),
    "mean_rejected_tokens": sum(len_rejected) / len(len_rejected),
    "n_pairs": len(pairs),
    "n_train": len(train_pairs),
    "n_heldout": len(heldout_pairs),
}

print()
print(f"รวมทั้งหมด {len(pairs)} คู่ | train {len(train_pairs)} | held-out {len(heldout_pairs)}")
print(f"ความยาวเฉลี่ย chosen   : {DATA_LEN_STATS['mean_chosen_tokens']:.1f} token")
print(f"ความยาวเฉลี่ย rejected : {DATA_LEN_STATS['mean_rejected_tokens']:.1f} token")
print()
print("จดตัวเลขสองบรรทัดบนไว้ให้ดี ถ้า chosen ยาวกว่า rejected อย่างเป็นระบบตั้งแต่ต้น")
print("สิ่งที่โมเดลอาจเรียนคือ 'ตอบยาวไว้ก่อน' ไม่ใช่ 'ตอบดี' -- นี่คือ length bias")
print()
print("ตัวอย่างคู่หนึ่ง:")
print("  chosen  :", pairs[0]["chosen"][:160].replace(chr(10), " "))
print("  rejected:", pairs[0]["rejected"][:160].replace(chr(10), " "))
print(f"  th_ratio chosen={th_ratio(pairs[0]['chosen']):.2f} "
      f"rejected={th_ratio(pairs[0]['rejected']):.2f}")


---

## 7. โค้ดหลัก (Main code)

### 7.1 เขียน DPO loss เองด้วยมือ

หัวใจของหัวข้อนี้ไม่ใช่การเรียกใช้ไลบรารี แต่คือการเขียน loss เอง
แล้วพิสูจน์ว่ามันตรงกับของจริง

สามจุดที่ต้องทำให้ถูก ไม่งั้นสมการ 3.4 จะไม่ใช่สมการ 3.4 อีกต่อไป:

1. **รวม log-prob เฉพาะ token ฝั่งคำตอบ** — token ของ prompt ต้องถูก mask เป็น `-100`
   ในทางทฤษฎี prompt เหมือนกันทั้งสองฝั่งจึงน่าจะตัดกัน แต่ในทางปฏิบัติ
   padding และความยาวที่ต่างกันทำให้มัน **ไม่ตัดกันพอดี** และ margin จะเพี้ยน
2. **reference ต้องอยู่ใต้ `torch.no_grad()`** — ถ้าลืม จะไม่มี error ใด ๆ
   แต่ VRAM จะพุ่งจนอาจ OOM และ reference จะไม่ได้ถูกแช่แข็งจริง
   นี่คือ bug ที่เงียบที่สุดของตอนนี้
3. **ใช้ผลรวม ไม่ใช่ค่าเฉลี่ย** — สมการเขียนไว้เป็น $\log\pi(y|x)$ ของทั้งลำดับ
   ซึ่งคือผลรวมของ log-prob รายตัว การหารด้วยความยาวคือ loss คนละตัว
   (และนี่เองคือที่มาของ length bias ที่เราจะวัดในหัวข้อ 9)

เซลล์ถัดไปมีทั้ง `seq_logp` และ `dpo_loss` ครบทั้ง DPO อยู่ในนั้นแล้ว ไม่มีอะไรซ่อนอยู่อีก
ท้ายเซลล์เป็น self-check ที่ไม่ต้องใช้โมเดลจริงเลย: ถ้า policy เท่ากับ reference เป๊ะ
แล้ว $\Delta = 0$ ดังนั้น loss ต้องเท่ากับ $-\log\sigma(0) = \ln 2$ พอดี
เป็นการตรวจว่าสูตรถูกก่อนจะเอาไปเจอโมเดลจริง


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.1 -- DPO loss เขียนเองด้วย PyTorch ล้วน ๆ
# -----------------------------------------------------------------------------
def seq_logp(model, input_ids, attention_mask, labels):
    """ผลรวม log-prob ของ 'เฉพาะส่วนคำตอบ' (token ของ prompt ถูก mask เป็น -100)"""
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    logits = logits[:, :-1, :]                   # ตำแหน่ง t ทำนาย token ที่ t+1
    target = labels[:, 1:]                       # จึงต้องเลื่อนเป้าหมายไป 1
    mask = target.ne(-100)                       # นับเฉพาะ token คำตอบ
    target = target.masked_fill(~mask, 0)        # กัน gather พังที่ตำแหน่ง -100
    logp = torch.log_softmax(logits.float(), dim=-1)
    tokp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return (tokp * mask).sum(-1)                 # [B] -- ผลรวม ไม่ใช่ค่าเฉลี่ย


def dpo_loss(policy, batch, beta=0.1):
    pi_w = seq_logp(policy, batch["chosen_ids"], batch["chosen_mask"], batch["chosen_labels"])
    pi_l = seq_logp(policy, batch["rejected_ids"], batch["rejected_mask"], batch["rejected_labels"])

    with torch.no_grad(), policy.disable_adapter():   # reference: ปิด adapter + ไม่เอา gradient
        ref_w = seq_logp(policy, batch["chosen_ids"], batch["chosen_mask"], batch["chosen_labels"])
        ref_l = seq_logp(policy, batch["rejected_ids"], batch["rejected_mask"], batch["rejected_labels"])

    delta = (pi_w - ref_w) - (pi_l - ref_l)                # Δ ในหัวข้อ 4
    loss = -F.logsigmoid(beta * delta).mean()              # สมการ 3.4 ตรงตัว

    r_w = beta * (pi_w - ref_w).detach()                   # implicit reward ของ chosen
    r_l = beta * (pi_l - ref_l).detach()                   # implicit reward ของ rejected
    return loss, r_w, r_l


# --- self-check ที่ไม่ต้องใช้โมเดลจริง -------------------------------------
# ถ้า policy == reference แล้ว delta = 0 ทุกคู่ ดังนั้น loss ต้องเท่ากับ ln 2 พอดี
class _FrozenLogits:
    """โมเดลปลอมที่คืน logits คงที่ ไม่ว่าจะเปิดหรือปิด adapter"""

    def __init__(self, table):
        self.table = table

    def __call__(self, input_ids=None, attention_mask=None):
        class _Out:
            pass

        out = _Out()
        out.logits = self.table[int(input_ids[0, 0])]
        return out

    class _Ctx:
        def __enter__(self):
            return None

        def __exit__(self, *exc):
            return False

    def disable_adapter(self):
        return _FrozenLogits._Ctx()


torch.manual_seed(SEED)
_V, _B, _T = 37, 3, 11
_ids_w = torch.randint(1, _V, (_B, _T))
_ids_l = torch.randint(1, _V, (_B, _T))
_ids_w[:, 0], _ids_l[:, 0] = 0, 1
_fake = _FrozenLogits({0: torch.randn(_B, _T, _V), 1: torch.randn(_B, _T, _V)})

_labels_w, _labels_l = _ids_w.clone(), _ids_l.clone()
_labels_w[:, :4] = -100          # mask ฝั่ง prompt
_labels_l[:, :4] = -100
_batch = {
    "chosen_ids": _ids_w, "chosen_mask": torch.ones_like(_ids_w), "chosen_labels": _labels_w,
    "rejected_ids": _ids_l, "rejected_mask": torch.ones_like(_ids_l), "rejected_labels": _labels_l,
}
_loss, _rw, _rl = dpo_loss(_fake, _batch, beta=0.1)

print(f"self-check | policy == reference -> loss = {_loss.item():.10f}")
print(f"self-check | ค่าที่ทฤษฎีบอก      -> ln 2 = {math.log(2):.10f}")
assert abs(_loss.item() - math.log(2)) < 1e-5, "สูตรผิดตั้งแต่ยังไม่เจอโมเดลจริง"
assert float(_rw.abs().max()) < 1e-5 and float(_rl.abs().max()) < 1e-5
print("self-check | implicit reward ทั้งสองฝั่งเป็นศูนย์พอดี -> ผ่าน")

# และตรวจว่าการ mask prompt เปลี่ยนค่าจริง ไม่ใช่โค้ดประดับ
_no_mask = seq_logp(_fake, _ids_w, torch.ones_like(_ids_w), _ids_w.clone())
_with_mask = seq_logp(_fake, _ids_w, torch.ones_like(_ids_w), _labels_w)
print(f"ไม่ mask prompt: {[round(float(v), 3) for v in _no_mask]}")
print(f"mask prompt   : {[round(float(v), 3) for v in _with_mask]}")
assert not torch.allclose(_no_mask, _with_mask)
print("=> การ mask prompt เปลี่ยนตัวเลขจริง จุดที่ 1 ในหัวข้อ 7.1 จึงไม่ใช่เรื่องเล็ก")


### 7.2 พิสูจน์ว่ามันตรงกับ TRL

ตอนนี้เราเอาสูตรที่เขียนเองไปเทียบกับ `DPOTrainer` ของ TRL บน **batch เดียวกันเป๊ะ**

มีสามอย่างที่ต้องจัดการก่อน ไม่งั้นการเปรียบเทียบจะไม่ยุติธรรมหรือไม่มีความหมาย:

**1. batch ต้องเป็นก้อนเดียวกันจริง ๆ** — เราจึงไม่สร้าง batch สองชุดแยกกัน
แต่ให้ collator ของ TRL สร้าง batch ขึ้นมาก้อนเดียว แล้ว **แปลง** ก้อนนั้น
เป็นรูปแบบที่สูตรของเราต้องการ ถ้าสร้างสองชุดแยกกัน ความต่างที่เจออาจมาจาก
การตัดข้อความหรือการ pad ไม่ตรงกัน ซึ่งไม่ได้บอกอะไรเกี่ยวกับสมการเลย

**2. ใช้ batch ขนาด 1** — เพราะ TRL จะจัดการ padding ฝั่งซ้ายด้วยวิธีของมันเอง
เมื่อมีหลายตัวอย่างในก้อนเดียว การใช้ขนาด 1 ตัดตัวแปรนั้นออกให้หมด
สิ่งที่เรากำลังพิสูจน์คือ **สูตร** ไม่ใช่วิธี pad

**3. คำนวณใน fp32** — สองสูตรที่ถูกต้องเท่ากันทางคณิตศาสตร์ ยังให้ค่าต่างกันได้
ระดับ 1e-3 ถ้าคำนวณ log-softmax ใน fp16 บนลำดับยาวหลายร้อย token
เพราะลำดับการปัดเศษต่างกัน เราจึง cast เป็น fp32 ชั่วคราวตอนพิสูจน์
แล้วค่อย cast กลับเป็น fp16 ตอนเทรน

**และข้อที่สำคัญที่สุด:** จำที่เซลล์ก่อนหน้าพิมพ์ไว้ไหมว่าตอนนี้ policy กับ reference
ให้ logits เท่ากันเป๊ะ เพราะ LoRA ถูก init ให้ $B = 0$
ถ้าเราพิสูจน์ตอนนี้เลย ทั้งสองฝั่งจะได้ $\ln 2$ เท่ากันพอดี **แม้สูตรของเราจะผิด**
การ assert แบบนั้นผ่านโดยไม่ได้พิสูจน์อะไรเลย

เราจึง **สุ่มค่า `lora_B` เล็ก ๆ ก่อน** เพื่อให้ $\Delta \neq 0$ จริง
แล้วค่อยเทียบ พอเทียบเสร็จจึงคืนค่ากลับเป็นศูนย์เพื่อให้การเทรนเริ่มจากจุดตั้งต้นมาตรฐาน
เซลล์จะพิมพ์ค่า $\Delta$ ออกมาให้เห็นว่ามันไม่ใช่ศูนย์


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.2 -- พิสูจน์ว่า loss ที่เขียนเอง == loss ของ TRL
# -----------------------------------------------------------------------------
from trl import DPOConfig, DPOTrainer

BETA = 0.1

cfg_dpo = DPOConfig(
    output_dir="dpo-out",
    beta=BETA,
    loss_type="sigmoid",              # ต้องตรงกับสูตรที่เราเขียนเอง
    label_smoothing=0.0,              # ถ้าไม่ศูนย์ สูตรจะไม่ใช่สมการ 3.4 อีกต่อไป
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch = 16
    num_train_epochs=2,
    learning_rate=5e-6,               # ต่ำกว่า SFT มาก -- ดูคำเตือนด้านล่าง
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_length=768,
    max_prompt_length=256,
    fp16=USE_FP16,                    # บน T4 ค่านี้คือ True (มาจาก Cell 0)
    bf16=SUPPORTS_BF16,               # บน T4 ค่านี้คือ False
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = DPOTrainer(
    model=policy,
    args=cfg_dpo,
    train_dataset=train_ds,
    processing_class=tokenizer,
)


def manual_batch_from_trl(b):
    """แปลง batch ของ TRL ให้เป็นรูปแบบที่สูตรของเราต้องการ -- ก้อนเดียวกันเป๊ะ"""
    p_ids, p_mask = b["prompt_input_ids"], b["prompt_attention_mask"]
    out = {}
    for side in ("chosen", "rejected"):
        c_ids, c_mask = b[f"{side}_input_ids"], b[f"{side}_attention_mask"]
        ids = torch.cat([p_ids, c_ids], dim=1)
        att = torch.cat([p_mask, c_mask], dim=1)
        loss_mask = torch.cat([torch.zeros_like(p_mask), c_mask], dim=1)
        out[f"{side}_ids"] = ids
        out[f"{side}_mask"] = att
        out[f"{side}_labels"] = ids.masked_fill(loss_mask == 0, -100)
    return out


row = trainer.train_dataset[0]
trl_batch = trainer.data_collator([row])
trl_batch = {k: (v.to(DEV) if torch.is_tensor(v) else v) for k, v in trl_batch.items()}
manual_batch = manual_batch_from_trl(trl_batch)

print("รูปร่างของ batch ที่ใช้พิสูจน์ (batch size = 1):")
for k in ("prompt_input_ids", "chosen_input_ids", "rejected_input_ids"):
    print(f"  {k:20s} {tuple(trl_batch[k].shape)}")
print(f"  token ฝั่งคำตอบที่นับจริง (chosen)  : "
      f"{int((manual_batch['chosen_labels'] != -100).sum())}")
print(f"  token ฝั่งคำตอบที่นับจริง (rejected): "
      f"{int((manual_batch['rejected_labels'] != -100).sum())}")

# ทำให้ policy != reference จริง ๆ ไม่งั้น assert จะผ่านแบบไม่มีความหมาย (ดูหัวข้อ 7.2)
lora_b_params = [p for n, p in policy.named_parameters() if "lora_B" in n]
saved_b = [p.data.clone() for p in lora_b_params]
torch.manual_seed(SEED)
for p in lora_b_params:
    p.data = torch.randn_like(p.data) * 0.01

policy.eval()                      # ปิด dropout ทุกชนิด -- ต้อง deterministic
orig_dtype = next(policy.parameters()).dtype
policy.float()                     # เทียบใน fp32 (ดูเหตุผลข้อ 3 ด้านบน)

with torch.no_grad():
    loss_manual, r_w, r_l = dpo_loss(policy, manual_batch, beta=BETA)
    loss_trl = trainer.compute_loss(policy, trl_batch)
    if isinstance(loss_trl, tuple):
        loss_trl = loss_trl[0]

delta_val = float((r_w - r_l).sum()) / BETA
print()
print(f"Δ (margin ก่อนคูณ beta) = {delta_val:+.6f}   <- ต้องไม่ใช่ศูนย์ ไม่งั้นการพิสูจน์ไร้ความหมาย")
print(f"implicit reward chosen  = {float(r_w.sum()):+.6f}")
print(f"implicit reward rejected= {float(r_l.sum()):+.6f}")
print()
print(f"loss ที่เราเขียนเอง : {loss_manual.item():.8f}")
print(f"loss จาก TRL       : {loss_trl.item():.8f}")
print(f"ผลต่าง             : {abs(loss_manual.item() - loss_trl.item()):.3e}")

assert abs(delta_val) > 1e-6, "policy ยังเท่ากับ reference -- การพิสูจน์จะไม่มีความหมาย"
assert torch.allclose(loss_manual, loss_trl, atol=1e-4)
print()
print("ตรงกัน:", loss_manual.item(), loss_trl.item())
print("=> สมการที่อนุมานมาทั้งหัวข้อ 3 ให้ค่าเท่ากับไลบรารีที่คนทั้งโลกใช้")

# คืนสภาพ: lora_B กลับเป็นศูนย์ และน้ำหนักกลับเป็น fp16 (แต่ adapter ต้องเป็น fp32)
for p, saved in zip(lora_b_params, saved_b):
    p.data = saved
policy.to(orig_dtype)
cast_trainable_to_fp32(policy)
policy.train()
print("\nคืนสภาพโมเดลเรียบร้อย พร้อมเทรนจริง")


> **ถ้า assert ไม่ผ่าน อย่าเพิ่งโทษโค้ดตัวเอง**
> สาเหตุที่พบบ่อยเรียงตามลำดับ: `label_smoothing` ไม่เป็นศูนย์,
> `loss_type` ไม่ใช่ `"sigmoid"`, batch ที่ป้อนสองฝั่งไม่ใช่ตัวอย่างเดียวกัน,
> หรือ padding/masking ไม่ตรงกัน ทั้งสี่ข้อคือความไม่ตรงกันของ **นิยาม** ไม่ใช่ bug
> และการไล่หามันคือบทเรียนที่ดีที่สุดของหัวข้อนี้

> **DPO ต้องการ learning rate ต่ำกว่า SFT มาก**
> SFT ด้วย LoRA ใช้ `2e-4` ได้สบาย แต่ DPO ที่ `2e-4` จะทำให้ policy วิ่งหนี reference
> ภายในไม่กี่สิบ step แล้วภาษาจะพังจนอ่านไม่ออก
> ใช้ **`5e-6`** เป็นจุดตั้งต้น เพราะ DPO ไม่ได้กำลังสอนเนื้อหาใหม่
> มันแค่ **เอียงการกระจายความน่าจะเป็นที่มีอยู่แล้ว** ซึ่งใช้แรงน้อยกว่ามาก

---

## 8. ผลลัพธ์ (Results)

รันเทรนจริง 2 epoch ด้วย $\beta = 0.1$, lr = 5e-6, effective batch = 16

### สิ่งที่จะทำให้คุณตกใจตอนดู log ครั้งแรก

ระหว่างเทรน คุณจะเห็น `rewards/chosen` และ `rewards/rejected` ใน log ของ TRL
และสิ่งที่เกิดขึ้นเกือบทุกครั้งคือ **ทั้งสองค่าไหลลงเป็นลบพร้อมกัน**
ขณะที่ `rewards/margins` กว้างขึ้นเรื่อย ๆ

**นี่คือเรื่องปกติ ไม่ใช่อาการพัง** จำนิยามไว้:
$\hat r = \beta\log(\pi_\theta/\pi_{\text{ref}})$ ดังนั้น $\hat r$ ติดลบแปลว่า
policy ให้ความน่าจะเป็นกับข้อความนั้น **น้อยกว่า** reference

DPO ไม่ได้ถูกสั่งให้ "ทำให้ chosen น่าจะเป็นมากขึ้น" มันถูกสั่งให้
**"ทำให้ช่องว่างกว้างขึ้น"** เท่านั้น การกด rejected ลงแรง ๆ แล้วกด chosen ลงเบา ๆ
ก็ตอบโจทย์ได้เหมือนกัน และมักเป็นทางที่ง่ายกว่า

สิ่งที่ต้องดูจึงเป็น **margin** กับ **held-out accuracy** ไม่ใช่ระดับสัมบูรณ์ของ reward
แต่ถ้า `rewards/chosen` ดิ่งลงลึกมาก (เช่น ต่ำกว่า −10) นั่นเริ่มเป็นสัญญาณว่า
โมเดลกำลังทิ้ง reference — ให้ลด LR หรือเพิ่ม $\beta$


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8 -- เทรนจริง
# -----------------------------------------------------------------------------
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
TRAIN_TIME_S = time.time() - t0
VRAM_PEAK_GB = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(f"\nเวลาเทรน  : {TRAIN_TIME_S / 60:.1f} นาที")
print(f"VRAM peak : {VRAM_PEAK_GB:.2f} GB")

LOG = [dict(h) for h in trainer.state.log_history]
last = [h for h in LOG if "rewards/chosen" in h]
if last:
    h = last[-1]
    print()
    print("ค่าสุดท้ายจาก log ของ TRL:")
    for k in ("rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies"):
        if k in h:
            print(f"  {k:20s} {h[k]:+.4f}")


---

## 9. เปรียบเทียบ (Comparison)

วัดสี่อย่าง ตามลำดับความสำคัญ:

1. **กราฟสี่เส้นมาตรฐานของ DPO** — `rewards/chosen`, `rewards/rejected`,
   `rewards/margins`, `rewards/accuracies` เพื่อยืนยันด้วยตาว่าที่อธิบายไว้ในหัวข้อ 8
   เกิดขึ้นจริงกับรันนี้
2. **Held-out preference accuracy พร้อม Wilson 95% CI** — สัดส่วนคู่ที่ implicit reward
   ของ chosen มากกว่า rejected **บนคู่ที่ไม่เคยเห็นตอนเทรน**
   accuracy บน train ที่สูงลิ่วแต่ held-out ไม่ขยับ แปลว่า overfit
3. **การกระจายของ margin ทั้ง distribution ไม่ใช่แค่ค่าเฉลี่ย** — ค่าเฉลี่ยที่สวยงาม
   อาจมาจากคู่ไม่กี่คู่ที่ margin สูงลิ่ว ขณะที่คู่ส่วนใหญ่ยังอยู่แถวศูนย์
   histogram บอกความจริงข้อนี้ ตัวเลขเดียวปิดบังมันไว้
4. **ความยาวคำตอบและ `th_ratio` ก่อน-หลัง** — เพราะ DPO มี **length bias**
   ที่รู้จักกันดี ถ้าความยาวเพิ่มขึ้นมาก ให้สงสัยไว้ก่อนว่าส่วนหนึ่งของ
   "คุณภาพที่ดีขึ้น" คือการตอบยาวขึ้นเฉย ๆ

ข้อดีของการใช้ LoRA ปรากฏอีกครั้งตรงนี้: การวัด "ก่อน" กับ "หลัง"
ทำได้ในเซลล์เดียวกันด้วยโมเดลตัวเดียวกัน เพราะ "ก่อน" คือแค่ปิด adapter
ไม่ต้องโหลดโมเดลใหม่ และไม่มีความเสี่ยงว่าสองการวัดจะอยู่คนละสภาพแวดล้อม


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.1 -- กราฟสี่เส้นมาตรฐานของ DPO
# -----------------------------------------------------------------------------
use_thai_font()

keys = ["rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies"]
titles = [
    "rewards/chosen (มักไหลลง -- ปกติ)",
    "rewards/rejected (ควรไหลลงแรงกว่า)",
    "rewards/margins (ตัวที่ต้องดูจริง ๆ)",
    "rewards/accuracies (สัดส่วนคู่ที่จัดอันดับถูก)",
]
colors = ["#2563eb", "#dc2626", "#16a34a", "#f59e0b"]

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2))
for ax, key, title, color in zip(axes.ravel(), keys, titles, colors):
    pts = [(h["step"], h[key]) for h in LOG if key in h]
    if pts:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], "-o", ms=3, color=color)
    ax.axhline(0, color="#94a3b8", lw=1, ls=":")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("optimizer step")
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle("สี่เส้นมาตรฐานของ DPO -- chosen และ rejected ไหลลงพร้อมกันคือเรื่องปกติ",
             fontsize=11)
fig.tight_layout()
fig.savefig("fig_dpo_curves.png", dpi=150, bbox_inches="tight")
plt.show()

loss_pts = [(h["step"], h["loss"]) for h in LOG if "loss" in h]
if loss_pts:
    print(f"loss step แรก = {loss_pts[0][1]:.4f}  (ควรใกล้ ln 2 = {math.log(2):.4f} "
          f"เพราะตอนเริ่ม policy == reference)")
    print(f"loss step สุดท้าย = {loss_pts[-1][1]:.4f}")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.2 -- held-out preference accuracy + การกระจายของ margin
# -----------------------------------------------------------------------------
MAX_LEN, MAX_PROMPT_LEN = 768, 256


def encode_pair(prompt, completion):
    """คืน (input_ids, labels) โดย mask ฝั่ง prompt เป็น -100 ตามกติกาหัวข้อ 7.1"""
    p_ids = tokenizer(prompt, add_special_tokens=False).input_ids[-MAX_PROMPT_LEN:]
    c_ids = tokenizer(completion, add_special_tokens=False).input_ids
    c_ids = (c_ids + [tokenizer.eos_token_id])[: MAX_LEN - len(p_ids)]
    ids = p_ids + c_ids
    labels = [-100] * len(p_ids) + c_ids
    return ids, labels


def collate_side(rows):
    """right-pad -- ตำแหน่ง pad ถูก mask ทั้งใน attention และใน labels"""
    width = max(len(r[0]) for r in rows)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(rows), width), pad, dtype=torch.long)
    att = torch.zeros((len(rows), width), dtype=torch.long)
    lab = torch.full((len(rows), width), -100, dtype=torch.long)
    for i, (r_ids, r_lab) in enumerate(rows):
        n = len(r_ids)
        ids[i, :n] = torch.tensor(r_ids, dtype=torch.long)
        att[i, :n] = 1
        lab[i, :n] = torch.tensor(r_lab, dtype=torch.long)
    return ids.to(DEV), att.to(DEV), lab.to(DEV)


policy.eval()
EVAL_BATCH = 4
margins = []
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(heldout_pairs), EVAL_BATCH):
        chunk = heldout_pairs[i:i + EVAL_BATCH]
        w_rows = [encode_pair(p["prompt"], p["chosen"]) for p in chunk]
        l_rows = [encode_pair(p["prompt"], p["rejected"]) for p in chunk]
        w_ids, w_att, w_lab = collate_side(w_rows)
        l_ids, l_att, l_lab = collate_side(l_rows)
        batch = {
            "chosen_ids": w_ids, "chosen_mask": w_att, "chosen_labels": w_lab,
            "rejected_ids": l_ids, "rejected_mask": l_att, "rejected_labels": l_lab,
        }
        _, r_w, r_l = dpo_loss(policy, batch, beta=BETA)
        margins.extend([float(v) for v in (r_w - r_l)])

n_correct = sum(1 for m in margins if m > 0)
n_total = len(margins)
lo, hi = wilson_ci(n_correct, n_total)
mean_margin = sum(margins) / n_total

PREF = {
    "n": n_total,
    "n_correct": n_correct,
    "accuracy": n_correct / n_total,
    "ci_low": lo,
    "ci_high": hi,
    "mean_margin": mean_margin,
    "median_margin": sorted(margins)[n_total // 2],
    "frac_margin_near_zero": sum(1 for m in margins if abs(m) < 0.01) / n_total,
}

print(f"วัด held-out {n_total} คู่ ใน {time.time() - t0:.0f} วินาที")
print(f"preference accuracy = {100 * PREF['accuracy']:.1f}% "
      f"[95% CI {100 * lo:.1f}-{100 * hi:.1f}]  ({n_correct}/{n_total})")
print(f"mean margin   = {mean_margin:+.4f}")
print(f"median margin = {PREF['median_margin']:+.4f}")
print(f"สัดส่วนคู่ที่ margin เกือบเป็นศูนย์ = {100 * PREF['frac_margin_near_zero']:.1f}%")
print()
print("ถ้า mean สวยแต่ median แถวศูนย์ แปลว่าค่าเฉลี่ยถูกลากด้วยคู่ไม่กี่คู่")
print("นั่นคือเหตุผลที่เราวาด histogram ไม่ใช่รายงานแค่ค่าเฉลี่ย")

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.hist(margins, bins=30, color="#2563eb", alpha=0.85)
ax.axvline(0, color="#dc2626", lw=1.5, ls="--", label="margin = 0")
ax.axvline(mean_margin, color="#16a34a", lw=1.5, label=f"mean = {mean_margin:+.3f}")
ax.set_xlabel("implicit reward margin (chosen − rejected)")
ax.set_ylabel("จำนวนคู่")
ax.set_title(f"การกระจายของ margin บน held-out {n_total} คู่\n"
             f"accuracy = {100 * PREF['accuracy']:.1f}% "
             f"[Wilson 95% CI {100 * lo:.1f}-{100 * hi:.1f}]", fontsize=11)
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_margin_hist.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.3 -- ความยาวคำตอบและ th_ratio ก่อน vs หลัง (length bias)
# "ก่อน" = ปิด adapter | "หลัง" = เปิด adapter -- โมเดลตัวเดียวกัน เงื่อนไขเดียวกัน
# -----------------------------------------------------------------------------
N_PROBE = 24
probe_prompts = [p["prompt"] for p in heldout_pairs[:N_PROBE]]


def generate_many(prompts, adapter_on, batch_size=8, max_new_tokens=128):
    tokenizer.padding_side = "left"
    outs = []
    ctx = torch.no_grad()
    try:
        with ctx:
            for i in range(0, len(prompts), batch_size):
                chunk = prompts[i:i + batch_size]
                enc = tokenizer(chunk, return_tensors="pt", padding=True,
                                truncation=True, max_length=320).to(DEV)
                torch.manual_seed(SEED)
                if adapter_on:
                    out = policy.generate(**enc, max_new_tokens=max_new_tokens,
                                          do_sample=False,
                                          pad_token_id=tokenizer.pad_token_id)
                else:
                    with policy.disable_adapter():
                        out = policy.generate(**enc, max_new_tokens=max_new_tokens,
                                              do_sample=False,
                                              pad_token_id=tokenizer.pad_token_id)
                new_tokens = out[:, enc["input_ids"].shape[1]:]
                outs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    finally:
        tokenizer.padding_side = "right"
    return outs


policy.eval()
t0 = time.time()
before_texts = generate_many(probe_prompts, adapter_on=False)
after_texts = generate_many(probe_prompts, adapter_on=True)
print(f"generate ก่อน/หลัง เสร็จใน {time.time() - t0:.0f} วินาที")


def summarize(texts):
    tok_lens = [len(tokenizer(t, add_special_tokens=False).input_ids) for t in texts]
    return {
        "n": len(texts),
        "mean_tokens": sum(tok_lens) / len(texts),
        "mean_chars": sum(len(t) for t in texts) / len(texts),
        "th_ratio": sum(th_ratio(t) for t in texts) / len(texts),
    }


LENGTH_STATS = {"before": summarize(before_texts), "after": summarize(after_texts)}
b, a = LENGTH_STATS["before"], LENGTH_STATS["after"]

print()
print(f"{'':10s} {'tokens':>9s} {'chars':>9s} {'th_ratio':>9s}")
print("-" * 40)
print(f"{'ก่อน DPO':10s} {b['mean_tokens']:9.1f} {b['mean_chars']:9.1f} {b['th_ratio']:9.3f}")
print(f"{'หลัง DPO':10s} {a['mean_tokens']:9.1f} {a['mean_chars']:9.1f} {a['th_ratio']:9.3f}")
print(f"{'ส่วนต่าง':10s} {a['mean_tokens'] - b['mean_tokens']:+9.1f} "
      f"{a['mean_chars'] - b['mean_chars']:+9.1f} {a['th_ratio'] - b['th_ratio']:+9.3f}")

ratio = a["mean_tokens"] / b["mean_tokens"] if b["mean_tokens"] else float("nan")
print()
print(f"ความยาวเปลี่ยนไป {100 * (ratio - 1):+.1f}%")
print("ถ้าเพิ่มขึ้นมาก ให้สงสัยไว้ก่อนว่าส่วนหนึ่งของ 'คุณภาพที่ดีขึ้น' คือ length bias")
print(f"(ในชุดข้อมูลของเรา chosen ยาวเฉลี่ย {DATA_LEN_STATS['mean_chosen_tokens']:.1f} token "
      f"ส่วน rejected ยาว {DATA_LEN_STATS['mean_rejected_tokens']:.1f} token)")
print()
print("ตัวอย่างคำตอบเดียวกัน:")
print("  ก่อน:", before_texts[0][:180].replace(chr(10), " "))
print("  หลัง:", after_texts[0][:180].replace(chr(10), " "))


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.4 -- วัด KobEval-TH ซ้ำด้วยสัญญาเดิม แล้วเทียบกับ baseline
# -----------------------------------------------------------------------------
policy.eval()
t0 = time.time()
after = evaluate(
    policy,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name=f"Qwen3-0.6B + DPO (β={BETA})",
    out_path="results_after.json",
    extra={"train_time_s": TRAIN_TIME_S, "vram_peak_gb": VRAM_PEAK_GB},
)
print(f"ใช้เวลาวัดผลหลัง DPO: {time.time() - t0:.0f} วินาที\n")

table = compare(baseline, after, out_path="results_table.json")

plot_before_after(
    baseline,
    after,
    slices=KOBEVAL_SLICES,
    title=f"ตอนที่ 4: ก่อน vs หลัง DPO (β={BETA})",
    out_path="fig_before_after.png",
    results_json="results_before_after.json",
)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.5 -- รวมทุกตัวเลขลง results.json ไฟล์เดียว เพื่อให้วิดเจ็ตบนบล็อกอ่านค่าจริง
# -----------------------------------------------------------------------------
from kobeval import write_results

payload = {
    "post": 4,
    "slug": "llm-04-dpo",
    "notebook": "04_dpo.ipynb",
    "model": MODEL_ID,
    "gpu": GPU_NAME,
    "supports_bf16": SUPPORTS_BF16,
    "eval_contract": EVAL_CONTRACT,
    "datasets": [
        "iapp/dpo_thai_tutorial",
        "airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k",
    ],
    "data": DATA_LEN_STATS,
    "hyperparameters": {
        "beta": BETA,
        "loss_type": "sigmoid",
        "label_smoothing": 0.0,
        "learning_rate": 5e-6,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.1,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 2,
        "max_length": 768,
        "max_prompt_length": 256,
        "lora_r": 16,
        "lora_alpha": 32,
        "fp16": USE_FP16,
    },
    "manual_vs_trl": {
        "loss_manual": float(loss_manual.item()),
        "loss_trl": float(loss_trl.item()),
        "abs_diff": abs(float(loss_manual.item()) - float(loss_trl.item())),
        "atol": 1e-4,
        "passed": True,
    },
    "training": {
        "train_time_s": TRAIN_TIME_S,
        "vram_peak_gb": VRAM_PEAK_GB,
        "log_history": LOG,
    },
    "preference": PREF,
    "generation": LENGTH_STATS,
    "kobeval": {
        "slices_run": KOBEVAL_SLICES,
        "before": {
            "model": baseline["model"],
            "accuracy": {k: v["accuracy"] for k, v in baseline["slices"].items()},
            "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in baseline["slices"].items()},
            "th_ratio": baseline["overall"]["th_ratio"],
        },
        "after": {
            "model": after["model"],
            "accuracy": {k: v["accuracy"] for k, v in after["slices"].items()},
            "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in after["slices"].items()},
            "th_ratio": after["overall"]["th_ratio"],
        },
        "mcnemar": table.get("mcnemar"),
        "markdown_table": table["markdown"],
    },
    "figures": [
        "fig_dpo_loss_and_gradient.png",
        "fig_dpo_curves.png",
        "fig_margin_hist.png",
        "fig_before_after.png",
    ],
}

path = write_results(payload, "results.json")
print(f"เขียนไฟล์แล้ว: {path}")
print(json.dumps({
    "manual_vs_trl": payload["manual_vs_trl"],
    "preference": payload["preference"],
    "generation": payload["generation"],
}, ensure_ascii=False, indent=2))


---

## 10. สรุป (Summary)

- **DPO ไม่ได้ประมาณ RLHF แต่แก้สมการเดียวกันในรูปปิด** — reward model กับ RL loop
  ตัดกันหายทางพีชคณิต และเรามี `assert` ในหัวข้อ 7.2 เป็นหลักฐาน ไม่ใช่คำกล่าวอ้าง
- **$Z(x)$ หายไปได้เพราะ Bradley-Terry สนใจแค่ผลต่างของ reward** นี่คือกุญแจของทั้งตอน
- **โมเดลภาษาเป็น reward model ของตัวเอง** ผ่าน
  implicit reward $\hat r = \beta\log(\pi_\theta/\pi_{\text{ref}})$
- **gradient ถ่วงน้ำหนักด้วยความผิดของตัวเอง** คู่ที่จัดอันดับถูกแล้วแทบไม่มี gradient
- **$\beta$ คือปุ่มแลกเปลี่ยน** ระหว่างตามใจ preference กับรักษาความสามารถเดิม
- **LR ต่ำมาก (5e-6)** เพราะเรากำลังเอียง distribution ไม่ได้สอนความรู้ใหม่
- **reward ทั้งสองฝั่งไหลลงพร้อมกันเป็นเรื่องปกติ** ดู margin อย่าดูระดับสัมบูรณ์
- **reference model ราคาศูนย์ไบต์** เมื่อเทรนด้วย LoRA — เป็นเหตุผลเชิงสถาปัตยกรรม
  ไม่ใช่แค่การประหยัดหน่วยความจำ
- **วัดความยาวคำตอบเสมอ** เพราะ length bias ปลอมตัวเป็นคุณภาพได้เนียนมาก

**ตอนต่อไป:** GRPO — เมื่อการจัดอันดับของที่มีอยู่ไม่พออีกต่อไป
เราจะให้โมเดลสุ่มคำตอบหลาย ๆ อันของตัวเองมาเปรียบเทียบกันเอง
โดยไม่ต้องมี value function แบบ PPO


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **DPO เป็น offline อย่างเคร่งครัด** — มันเรียนจากคู่คำตอบที่มีอยู่แล้วในไฟล์เท่านั้น
   สิ่งที่มันทำได้คือ **จัดลำดับพฤติกรรมที่โมเดลสุ่มออกมาได้อยู่แล้วใหม่**
   มันไม่มีทาง **ค้นพบ** วิธีตอบที่โมเดลฐานไม่เคยผลิตออกมาเลย
   ช่องว่างตรงนี้คือเหตุผลที่ตอน GRPO ต้องมีอยู่

2. **500 คู่คือการสาธิตกลไก ไม่ใช่การ align จริง** — งาน alignment ระดับใช้งานจริง
   ใช้คู่ preference ระดับหมื่นถึงแสนคู่ ต่างกันหลาย order of magnitude
   อย่าเอาผลนี้ไปอ้างว่าได้โมเดลไทยที่ดีกว่าเดิม

3. **`chosen` ไม่ได้มาจากการตัดสินของมนุษย์ทั้งหมด** — 400 คู่จาก 500 ใช้คำตอบอ้างอิง
   ของชุดข้อมูลเป็น `chosen` โดยอัตโนมัติ ซึ่งเป็น **การสมมติ** ว่าคำตอบอ้างอิงดีกว่า
   คำตอบของโมเดลฐานเสมอ สมมติฐานนี้ผิดได้ และเราไม่ได้ตรวจทีละคู่

4. **ขนาดชุดวัดผลเล็กมาก** — KobEval-TH มี slice ละ 30 ข้อ และ held-out preference
   มีราว 75 คู่ ช่วงความเชื่อมั่นจึงกว้าง ความต่างระดับ 1-2 ข้อ **ไม่ใช่**
   ความต่างที่มีนัยสำคัญ ให้ดูค่า p จาก McNemar ที่ `compare()` พิมพ์ออกมาด้วยเสมอ

5. **รัน $\beta$ ค่าเดียว** — บทความเปรียบเทียบ $\beta = 0.1$ กับ $0.5$
   แต่โน้ตบุ๊กนี้รันแค่ `0.1` เพื่อให้จบใน 20 นาทีบน T4 ฟรี
   ถ้าอยากได้ตารางเต็ม ให้แก้ `BETA` แล้วรันหัวข้อ 7.2 เป็นต้นไปซ้ำ
   (ใช้เวลาเพิ่มอีกราว 10 นาทีต่อค่า)

6. **รันครั้งเดียว ไม่ได้ทำ multiple seeds** — ใช้ greedy decoding และ seed=42 ตายตัว
   เพื่อให้ทำซ้ำได้ แต่ไม่ได้รายงานความแปรปรวนจากการเทรนหลาย seed

7. **การพิสูจน์ในหัวข้อ 7.2 ทำใน fp32 และ batch ขนาด 1** — เพราะสิ่งที่พิสูจน์คือ
   **ความถูกต้องของสูตร** ไม่ใช่ความเท่ากันของทุก kernel ใน fp16
   ตอนเทรนจริงเราใช้ fp16 ตามข้อจำกัดของ T4 ซึ่งจะมีความคลาดเคลื่อนเชิงตัวเลข
   ที่ใหญ่กว่า 1e-4 อยู่แล้วโดยธรรมชาติ

8. **ฮาร์ดแวร์จำกัด** — ทุกอย่างถูกบีบให้รันได้บน T4 ฟรี ซึ่งแปลว่าต้องใช้ fp16 แทน bf16,
   ใช้ sdpa แทน FlashAttention-2, batch เล็ก และ sequence สั้น
   ผลบนฮาร์ดแวร์ที่ใหญ่กว่าอาจต่างออกไป

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต Apache-2.0 — [kobkrit.com](https://kobkrit.com)*
